<a href="https://colab.research.google.com/github/NureTopchiiDaria/gen_ai_lab1_topchii/blob/main/LLM_Finetuning_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Copyright 2026 NURE Generative Neural Networks and their applications. All Rights Reserved.
#
# This notebook is based on Lab3 from
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of NURE Generative Neural Networks
# and their applications must reference:
#
# @ NURE Generative Neural Networks and their applications
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com


# Laboratory 1: Large Language Model (LLM) Fine-tuning

In this lab, you will fine-tune a multi-billion parameter large language model (LLM). The lab covers tokenization, templates, fine-tuning, and evaluation of a language model's style.

In [2]:
# Install and import MIT Deep Learning utilities
!pip install mitdeeplearning
import mitdeeplearning as mdl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 49.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.9/155.9 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from lion_pytorch import Lion
from itertools import islice

SEED = 42

def fix_seed(seed):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fix_seed(SEED)


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Part 1: Fine-tuning an LLM for style

We use Liquid AI LFM2-1.2B as the base language model.

## 1.1: Templating and tokenization

In [4]:
# Basic question-answer template
template_without_answer = "<|startoftext|><|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"
template_with_answer = template_without_answer + "{answer}<|im_end|>
"

print(template_with_answer.format(question="What is your name?", answer="My name is Lili!"))


SyntaxError: unterminated string literal (detected at line 2) (2637290085.py, line 2)

In [ ]:
# Load the tokenizer for Liquid AI LFM2-1.2B
model_id = "LiquidAI/LFM2-1.2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f"Vocab size: {len(tokenizer.get_vocab())}")


In [ ]:
text = "Here is some sample text!"
print(f"Original text: {text}")

tokens = tokenizer.encode(text, return_tensors="pt")
print(f"Encoded tokens: {tokens}")

decoded_text = tokenizer.decode(tokens[0], skip_special_tokens=True)
print(f"Decoded text: {decoded_text}")


In [ ]:
prompt = template_without_answer.format(question="What is the capital of France? Use one word.")
print(prompt)


## 1.2: Getting started with the LLM

In [ ]:
# Load the model -- note that this may take a few minutes
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")


In [ ]:
question = "What is the capital of France? Use one word."
prompt = template_without_answer.format(question=question)
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model(tokens)
    probs = F.softmax(output.logits, dim=-1)

next_token = torch.argmax(probs[0, -1, :]).item()
next_token_text = tokenizer.decode(next_token)
print(f"Prompt: {prompt}")
print(f"Predicted next token: {next_token_text}")


In [ ]:
prompt = template_without_answer.format(question="What does MIT stand for?")
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
output = model.generate(tokens, max_new_tokens=20)
print(tokenizer.decode(output[0]))


## 1.3: Monitoring with Comet

Add your own Comet/Opik API key and workspace below. The API keys visible in the PDF were replaced with placeholders in this reconstructed notebook.

In [ ]:
import opik
from opik import opik_context

OPIK_API_KEY = "tZ1n9crmsZtAPIipUq2iy3lqd"
os.environ["OPIK_API_KEY"] = OPIK_API_KEY
assert OPIK_API_KEY != "" and OPIK_API_KEY != "YOUR_OPIK_API_KEY"

os.environ["OPIK_PROJECT_NAME"] = "proj1"
os.environ["OPIK_WORKSPACE"] = "lab1-llm-finetuning"

opik.configure()
opik_client = opik.Opik()


## 1.4: Fine-tuning

In [ ]:
# train_loader, test_loader = mdl.lab3.create_dataloader(style="leprechaun")
# sample = train_loader.dataset[44]
# question = sample['instruction']
# answer = sample['response']
# answer_style = sample['response_style']
# print(f"Question: {question}

" +
#       f"Original Answer: {answer}

" +
#       f"Answer Style: {answer_style}")


### 1.4.1: Chat function

In [ ]:
def chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):
    prompt = template_without_answer.format(question=question)
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

    full_tokens = outputs[0]
    prompt_tokens = int(input_ids["input_ids"].shape[-1])
    completion_tokens = int(full_tokens.shape[-1] - prompt_tokens)
    output_tokens = full_tokens[prompt_tokens:] if only_answer else full_tokens
    result = tokenizer.decode(output_tokens, skip_special_tokens=True)

    return {
        "answer": result,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
        "prompt": prompt,
    }

answer = chat("What is the capital of Ireland?", only_answer=True, max_new_tokens=32)
print(answer["answer"])


### 1.4.2: Parameter-efficient fine-tuning

In [ ]:
def apply_lora(model, r=8, lora_alpha=16, lora_dropout=0.05):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "o_proj", "k_proj", "v_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    )
    lora_model = get_peft_model(model, lora_config)
    return lora_model

model = apply_lora(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"number of trainable parameters: {trainable_params}")
print(f"total parameters: {total_params}")
print(f"percentage of trainable parameters: {trainable_params / total_params * 100:.2f}%")


In [ ]:
def create_yoda_dataloaders(batch_size=16, seed=42, train_ratio=0.7, val_ratio=0.15):
    ds = load_dataset("databricks/databricks-dolly-15k", split="train")
    with open("data/text_styles/yoda.txt", "r", encoding="utf-8") as f:
        yoda_responses = [line.strip().replace("\n", "
") for line in f]

    n = len(yoda_responses)
    ds = ds.select(range(n))
    ds = ds.map(lambda x, idx: {"response_style": yoda_responses[idx]}, with_indices=True)
    ds = ds.shuffle(seed=seed)

    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_ds = ds.select(range(0, n_train))
    val_ds = ds.select(range(n_train, n_train + n_val))
    test_ds = ds.select(range(n_train + n_val, n))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


### 1.4.3: Forward pass and loss computation

In [ ]:
def forward_and_compute_loss(model, tokens, mask, context_length=512):
    tokens = tokens[:, :context_length]
    mask = mask[:, :context_length]

    x = tokens[:, :-1]
    y = tokens[:, 1:]
    mask = mask[:, 1:]

    logits = model(x).logits
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        y.view(-1),
        reduction="none"
    )
    loss = loss[mask.view(-1)].mean()
    return loss


### 1.4.4: Training loop for fine-tuning

In [ ]:
def train_step(step, model, optimizer, batch, tokenizer, context_length=512):
    losses = []
    last_ids = None
    for question, answer in zip(batch["instruction"], batch["response_style"]):
        text = template_with_answer.format(question=question, answer=answer)
        ids = tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
        mask = ids["offset_mapping"][:, :, 0] >= text.index(answer)
        loss = forward_and_compute_loss(model=model, tokens=ids["input_ids"], mask=mask, context_length=context_length)
        losses.append(loss)
        last_ids = ids

    batch_loss = torch.stack(losses).mean()
    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

    loss_value = float(batch_loss.item())
    return loss_value, batch["instruction"][0], batch["response_style"][0], last_ids

@opik.track(name="training_run")
def train(model, dataloader, tokenizer, val_loader=None, n_epochs=1,
          max_steps=200, context_length=512, learning_rate=1e-4,
          preview_every=25, run_name="run"):
    opik_context.update_current_trace(
        metadata={
            "run_name": run_name,
            "model": getattr(model, "name_or_path", "unknown"),
            "max_steps": max_steps,
            "n_epochs": n_epochs,
        },
        tags=[run_name],
    )

    rolling_losses = []
    optimizer = Lion(model.parameters(), lr=learning_rate)
    global_step = 0

    for epoch in range(n_epochs):
        model.train()
        for batch in dataloader:
            with opik.start_as_current_span("train_step"):
                loss_value, question, answer, ids = train_step(
                    global_step, model, optimizer, batch, tokenizer, context_length
                )
                rolling_losses.append(loss_value)
                opik_context.update_current_span(
                    input={"question": question},
                    output={"loss": loss_value},
                    metadata={"phase": "train", "step": global_step,
                              "epoch": epoch,
                              "tokens": int(ids["input_ids"].shape[-1]),
                              "answer_chars": len(answer)},
                )

            if global_step % preview_every == 0:
                with opik.start_as_current_span("preview_generation", type="llm"):
                    preview = chat("What is the capital of France?", only_answer=True)
                    opik_context.update_current_span(
                        input={"question": "What is the capital of France?"},
                        output={"answer": preview["answer"]},
                        metadata={"loss": f"{np.mean(rolling_losses):.4f}"},
                    )
                    print(f"[epoch {epoch} step {global_step}] loss: {np.mean(rolling_losses):.4f} | {preview['answer']}")
                    rolling_losses = []

            global_step += 1
            if global_step >= max_steps:
                break

        if val_loader is not None:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for val_batch in val_loader:
                    for q, a in zip(val_batch["instruction"], val_batch["response_style"]):
                        text = template_with_answer.format(question=q, answer=a)
                        ids = tokenizer(text, return_tensors="pt",
                                        return_offsets_mapping=True).to(model.device)
                        mask = ids["offset_mapping"][:, :, 0] >= text.index(a)
                        v_loss = forward_and_compute_loss(
                            model=model, tokens=ids["input_ids"],
                            mask=mask, context_length=context_length
                        )
                        val_losses.append(float(v_loss.item()))
            print(f"[epoch {epoch}] val_loss: {np.mean(val_losses):.4f}")

        if global_step >= max_steps:
            break

    opik_client.flush()
    return model

In [ ]:
print(chat("What is a good story about tennis", only_answer=True, max_new_tokens=200)["answer"])


# Part 2: Evaluating a style-tuned LLM

The goal is to fine-tune the model to generate responses in Yoda style, use an LLM judge to evaluate outputs, and use that information to improve the model.

## 2.1: Fine-tune well, you must!

In [ ]:
fix_seed(SEED)
train_loader, val_loader, test_loader = create_yoda_dataloaders(batch_size=16, seed=SEED)

model = train(
    model,
    train_loader,
    tokenizer,
    val_loader=val_loader,
    n_epochs=3,
    max_steps=200,
    context_length=512,
    learning_rate=2e-4,
    preview_every=25,
    run_name=f"yoda_seed_{SEED}"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
system_prompt = """
You are an impartial judge that evaluates if text was written by {style}.
An example piece of text from {style} is:
{example}
Now, analyze some new text carefully and respond on if it follows the
same style of {style}. Be critical to identify any issues in the text.
Then convert your feedback into a number between 0 and 10: 10 if the text
is written exactly in the style of {style}, 5 if mixed faithfulness to the
style, or 0 if the text is not at all written in the style of {style}.
Directly answer with the score formatted in a dictionary.
The format of your response should only be the dictionary and nothing else:
{{"score": <score between 0 and 10>}}
"""

system_prompt_strict = """
You are a strict evaluator of Yoda-style speech.
Score the text from 0 to 10:
10 means the text strongly uses Yoda-like inverted word order, short wise phrases, and Star-Wars-like tone.
5 means the text has some Yoda-like features but is inconsistent.
0 means the text is normal English and not Yoda-like.
Return only a JSON dictionary:
{"score": <score between 0 and 10>}
"""

style = "Yoda"
example = "The very Republic is threatened, if involved the Sith are. Hard to see, the dark side is. Discover who this assassin is, we must."
system_prompt = system_prompt.format(style=style, example=example)
print("=== System prompt ===")
print(system_prompt)


## 2.2: Setting up the judge LLM

In [ ]:
OPENROUTER_API_KEY = "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"
assert OPENROUTER_API_KEY != "" and OPENROUTER_API_KEY != "YOUR_OPENROUTER_API_KEY"

model_name = "qwen/qwen3-next-80b-a3b-instruct"
llm = mdl.lab3.LLMClient(model=model_name, api_key=OPENROUTER_API_KEY)


## 2.3: Defining the evaluation metric

In [ ]:
from opik.evaluation.metrics import base_metric, score_result

class LLMJudgeEvaluator(base_metric.BaseMetric):
    def __init__(self, judge: mdl.lab3.LLMClient = None, system_prompt: str = None):
        self.judge = judge
        self.system_prompt = system_prompt
        self.prompt_template = "Evaluate this text: {text}"

    def score(self, text: str, n_tries=20, **kwargs):
        """Evaluate by asking an LLM to score it."""
        for attempt in range(n_tries):
            try:
                prompt = self.prompt_template.format(text=text)
                with opik.start_as_current_span("judge_call"):
                    res = self.judge.ask(system=self.system_prompt, user=prompt, max_tokens=20)
                    opik_context.update_current_span(
                        input={"system": self.system_prompt, "user": prompt},
                        output={"judge_raw_response": res.choices[0].message.content},
                        metadata={"phase": "judge test"},
                        model=model_name,
                        provider="openrouter",
                    )

                res = res.choices[0].message.content
                res_dict = json.loads(res)
                max_score = 10
                score = res_dict["score"] / max_score
                score = max(0.0, min(score, 1.0))
                return score_result.ScoreResult(name="StyleScore", value=score)
            except Exception as e:
                if attempt == n_tries - 1:
                    raise e
                continue

judge = LLMJudgeEvaluator(llm, system_prompt=system_prompt)
judge_strict = LLMJudgeEvaluator(llm, system_prompt=system_prompt_strict)


## 2.4: Evaluating the model by scoring with your judge LLM

In [ ]:
def scoring_function(text):
    return judge.score(text).value

test_texts = [
    "Tennis is a fun sport. But you must concentrate.",
    "Fun sport, tennis is. But work hard, you must.",
    "Hard to see, the dark side is."
]

@opik.track(name="scoring first 3 samples")
def score_test_texts(test_texts):
    for text in test_texts:
        score = scoring_function(text)
        print(f"{text} ==> Score: {score}")

score_test_texts(test_texts)


In [ ]:
def generate_samples_from_test(test_loader, num_samples):
    samples = []
    for test_sample in tqdm(islice(test_loader, num_samples), total=num_samples):
        test_question = test_sample['instruction'][0]
        with torch.no_grad():
            generated = chat(test_question, only_answer=True, max_new_tokens=48, temperature=0.3)["answer"]
        samples.append(generated)
    return samples

fix_seed(SEED)
n_samples = 20
generated_samples = generate_samples_from_test(test_loader, num_samples=n_samples)


In [ ]:
base_samples = []
style_samples = []
for batch in test_loader:
    for response, response_style in zip(batch["response"], batch["response_style"]):
        base_samples.append(response)
        style_samples.append(response_style)
    if len(base_samples) >= n_samples:
        break

base_samples = base_samples[:n_samples]
style_samples = style_samples[:n_samples]


In [ ]:
@opik.track(name="evaluation batch track")
def evaluate_samples(samples, kind):
    scores = []
    for idx, text in enumerate(samples):
        with opik.start_as_current_span(f"score_{idx}"):
            score = judge.score(text).value
            opik_context.update_current_span(
                input={"kind": kind, "text_preview": text[:300]},
                output={"score": float(score)},
                metadata={"phase": "eval", "sample_index": idx},
            )
            scores.append(score)

    opik_context.update_current_trace(
        input={"kind": kind, "num_samples": len(samples)},
        output={"mean_score": float(np.mean(scores)), "std_score": float(np.std(scores))},
    )
    return scores

base_scores = evaluate_samples(base_samples, "base from test")
generated_scores = evaluate_samples(generated_samples, "generated")
style_scores = evaluate_samples(style_samples, "style from test")

print(f"Base : {np.mean(base_scores):.2f} ± {np.std(base_scores):.2f}")
print(f"Gen : {np.mean(generated_scores):.2f} ± {np.std(generated_scores):.2f}")
print(f"Train : {np.mean(style_scores):.2f} ± {np.std(style_scores):.2f}")
opik_client.flush()


In [ ]:
def compute_style_metrics(base_scores, generated_scores):
    base_scores = np.array(base_scores)
    generated_scores = np.array(generated_scores)
    recall = np.mean(generated_scores)
    specificity = np.mean(1 - base_scores)
    balanced_accuracy = 0.5 * (recall + specificity)
    style_gain = np.mean(generated_scores) - np.mean(base_scores)
    return {
        "Recall": recall,
        "BalancedAccuracy": balanced_accuracy,
        "StyleGain": style_gain,
    }

metrics = compute_style_metrics(base_scores, generated_scores)
metrics


In [ ]:
# Порівняння двох суддів: загальний vs строгий
def evaluate_samples_with_judge(samples, kind, evaluator):
    return [evaluator.score(t).value for t in tqdm(samples, desc=kind)]

gen_scores_strict  = evaluate_samples_with_judge(generated_samples, "gen_strict",  judge_strict)
base_scores_strict = evaluate_samples_with_judge(base_samples,      "base_strict", judge_strict)

print("=== Порівняння суддів (на generated_samples) ===")
print(f"  judge        (lenient) : {np.mean(generated_scores):.3f} ± {np.std(generated_scores):.3f}")
print(f"  judge_strict (strict)  : {np.mean(gen_scores_strict):.3f} ± {np.std(gen_scores_strict):.3f}")

metrics_strict = compute_style_metrics(base_scores_strict, gen_scores_strict)
print("\n=== Метрики зі строгим суддею ===")
for k, v in metrics_strict.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({
    'Score': [*base_scores, *generated_scores, *style_scores],
    'Type': ['Base']*len(base_scores) + ['Generated']*len(generated_scores) + ['Style']*len(style_scores)
})

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Score', hue='Type', multiple='dodge', stat='probability', bins=6)
plt.xlabel("Judge score")
plt.ylabel("Probability")
plt.title("Distribution of judge scores")
plt.show()


In [ ]:
labels = ["Base", "Generated", "Style"]
means = [np.mean(base_scores), np.mean(generated_scores), np.mean(style_scores)]
stds = [np.std(base_scores), np.std(generated_scores), np.std(style_scores)]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, means, yerr=stds, capsize=6)
plt.ylim(0, 1)
plt.ylabel("Judge score")
plt.title("Judge LLM scores by text type")
for bar, value in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
results = []
results.append({
    "seed": SEED,
    "learning_rate": 2e-4,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "context_length": 512,
    "temperature": 0.3,
    **metrics
})
results_df = pd.DataFrame(results)
results_df


In [ ]:
best_model_path = "best_yoda_lora"
model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"Saved LoRA weights to {best_model_path}")

base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
loaded_model = PeftModel.from_pretrained(base_model, best_model_path)
loaded_model.eval()
model = loaded_model


## 2.5: Monitoring with evals

## 2.5.1: Покращення

### 1. Lion optimizer замість AdamW
Використали Lion замість стандартного AdamW, бо він споживає менше пам'яті (не зберігає другий момент градієнта) і краще працює з трансформерами на малих learning rate. На практиці val_loss сходився стабільніше і швидше.

### 2. Два prompt templates для судді
Додали строгий промпт `system_prompt_strict`, який перевіряє конкретні ознаки
стилю Йоди: інвертований порядок слів, короткі фрази, тон Star Wars.
Порівняння lenient vs strict показує, чи модель справді навчилась стилю, чи просто видає схожі тексти.

### 3. Validation після кожної епохи
Додали підрахунок val_loss після кожної епохи, щоб бачити перенавчання.Якщо train_loss падає, а val_loss росте, то модель перенавчається і треба зупинитись.

In [ ]:
@opik.track(name="inference chat track")
def inference_chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):
    with opik.start_as_current_span("generate step"):
        gen = chat(question, max_new_tokens=max_new_tokens, temperature=temperature, only_answer=only_answer)
        opik_context.update_current_span(
            input={"question": question, "temperature": temperature, "max_new_tokens": max_new_tokens, "only_answer": only_answer},
            output={"answer": gen["answer"]},
            metadata={"phase": "inference"},
            usage={"prompt_tokens": gen["prompt_tokens"], "completion_tokens": gen["completion_tokens"], "total_tokens": gen["total_tokens"]},
            model=model_id,
            provider="huggingface",
        )

    with opik.start_as_current_span("style_score"):
        style_score = scoring_function(gen["answer"])
        opik_context.update_current_span(
            input={"text_preview": gen["answer"][:300]},
            output={"style_score": float(style_score)},
            metadata={"phase": "eval"},
        )

    opik_context.update_current_trace(
        input={"question": question},
        output={"answer": gen["answer"]},
        metadata={"temperature": temperature, "max_new_tokens": max_new_tokens, "only_answer": only_answer},
        feedback_scores=[{"name": "Yoda style eval", "value": float(style_score)}],
    )
    return gen["answer"]

answer = inference_chat(
    "Who was the only non-Jedi to wield a lightsaber in the original Star Wars trilogy?",
    only_answer=True,
    max_new_tokens=32,
)
print(answer)


In [ ]:
# Пункт б): 3 повтори одного і того ж конфігу з різними seed
REPEAT_CFG = {"lr": 2e-4, "rank": 8, "alpha": 16, "dropout": 0.05,
              "context": 512, "temperature": 0.3}
REPEAT_SEEDS = [42, 43, 44]

seed_repeat_results = []
for s in REPEAT_SEEDS:
    fix_seed(s)
    _model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    _model = apply_lora(_model, r=REPEAT_CFG["rank"],
                        lora_alpha=REPEAT_CFG["alpha"],
                        lora_dropout=REPEAT_CFG["dropout"])
    _train_loader, _val_loader, _test_loader = create_yoda_dataloaders(
        batch_size=16, seed=s)
    _model = train(
        _model, _train_loader, tokenizer,
        val_loader=_val_loader, n_epochs=2,
        max_steps=200, context_length=REPEAT_CFG["context"],
        learning_rate=REPEAT_CFG["lr"], preview_every=50,
        run_name=f"seed_repeat_{s}"
    )

    _prev_model = model
    model = _model

    fix_seed(s)
    _gen = generate_samples_from_test(_test_loader, n_samples)
    model = _prev_model

    _base, _style = [], []
    for _b in _test_loader:
        for r, rs in zip(_b["response"], _b["response_style"]):
            _base.append(r); _style.append(rs)
        if len(_base) >= n_samples:
            break
    _base_scores = evaluate_samples(_base[:n_samples], f"base_seed{s}")
    _gen_scores  = evaluate_samples(_gen,              f"gen_seed{s}")
    _m = compute_style_metrics(_base_scores, _gen_scores)
    seed_repeat_results.append({"seed": s, **_m})

seed_df = pd.DataFrame(seed_repeat_results)
print("=== Результати по трьох seed ===")
print(seed_df.to_string(index=False))
print()
print("=== Усереднені метрики (пункт б) ===")
print(seed_df[["Recall","BalancedAccuracy","StyleGain"]].mean().to_string())

In [ ]:
# Демонстрація: 5 прикладів до і після fine-tuning
demo_questions = [
    "What is the meaning of life?",
    "How do you make tea?",
    "What is artificial intelligence?",
    "Why is the sky blue?",
    "What makes a good leader?",
]

print(f"{'Question':<40} {'Answer (fine-tuned)':<60} {'Score':>6}")
print("-" * 110)
for q in demo_questions:
    ans = chat(q, only_answer=True, max_new_tokens=48, temperature=0.3)["answer"]
    score = judge.score(ans).value
    print(f"{q:<40} {ans[:58]:<60} {score:>6.2f}")

In [ ]:
experiments = [
    {"seed": 42, "lr": 1e-4, "rank": 8, "alpha": 16, "dropout": 0.05, "context": 512, "temperature": 0.3},
    {"seed": 43, "lr": 2e-4, "rank": 8, "alpha": 16, "dropout": 0.05, "context": 512, "temperature": 0.3},
    {"seed": 44, "lr": 3e-4, "rank": 8, "alpha": 16, "dropout": 0.05, "context": 512, "temperature": 0.3},
    {"seed": 45, "lr": 2e-4, "rank": 16, "alpha": 32, "dropout": 0.05, "context": 512, "temperature": 0.3},
    {"seed": 46, "lr": 2e-4, "rank": 16, "alpha": 64, "dropout": 0.05, "context": 512, "temperature": 0.3},
    {"seed": 47, "lr": 2e-4, "rank": 16, "alpha": 32, "dropout": 0.10, "context": 512, "temperature": 0.3},
    {"seed": 48, "lr": 2e-4, "rank": 16, "alpha": 32, "dropout": 0.05, "context": 1024, "temperature": 0.3},
    {"seed": 49, "lr": 2e-4, "rank": 16, "alpha": 32, "dropout": 0.05, "context": 512, "temperature": 0.7},
]

all_results = []
for cfg in experiments:
    fix_seed(cfg["seed"])
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    model = apply_lora(model, r=cfg["rank"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"])
    train_loader, val_loader, test_loader = create_yoda_dataloaders(batch_size=16, seed=cfg["seed"])
    model = train(
        model,
        train_loader,
        tokenizer,
        val_loader=val_loader,
        n_epochs=2,
        max_steps=300,
        context_length=cfg["context"],
        learning_rate=cfg["lr"],
        preview_every=50,
        run_name=f"yoda_seed_{cfg['seed']}"
    )

    fix_seed(cfg["seed"])
    n_samples = 20
    generated_samples = generate_samples_from_test(test_loader, n_samples)

    base_samples = []
    style_samples = []
    for batch in test_loader:
        for response, response_style in zip(batch["response"], batch["response_style"]):
            base_samples.append(response)
            style_samples.append(response_style)
        if len(base_samples) >= n_samples:
            break

    base_scores = evaluate_samples(base_samples[:n_samples], "base")
    generated_scores = evaluate_samples(generated_samples, "generated")
    style_scores = evaluate_samples(style_samples[:n_samples], "style")
    metrics = compute_style_metrics(base_scores, generated_scores)
    gen_scores_strict_loop  = [judge_strict.score(t).value for t in generated_samples]
    base_scores_strict_loop = [judge_strict.score(t).value for t in base_samples[:n_samples]]
    metrics_strict_loop = compute_style_metrics(base_scores_strict_loop, gen_scores_strict_loop)

    row = {
        "seed": cfg["seed"],
        "learning_rate": cfg["lr"],
        "lora_rank": cfg["rank"],
        "lora_alpha": cfg["alpha"],
        "lora_dropout": cfg["dropout"],
        "context_length": cfg["context"],
        "temperature": cfg["temperature"],
        **{f"{k}_lenient": v for k, v in metrics.items()},
        **{f"{k}_strict":  v for k, v in metrics_strict_loop.items()},
    }
    all_results.append(row)

results_df = pd.DataFrame(all_results)
results_df



## 2.6: Conclusion

Run the following cell before submitting your entry to the lab. Do not modify it.

In [ ]:
# DO NOT CHANGE/MODIFY THIS CELL.
# EXECUTE IT BEFORE SUBMITTING YOUR ENTRY TO THE LAB.
yoda_test_text = mdl.lab3.yoda_test_text
tokens = tokenizer(yoda_test_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**tokens)
    logits = outputs.logits[:, :-1]
    targets = tokens.input_ids[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

print(f"Cross entropy loss on held-out Yoda text: {loss.item():.4f}")
